# Colab Transcription Notebook

This notebook is the primary workflow for this repository.
It is designed for Google Colab and uses a stronger pre-correction transcription pipeline for Korean audio.

Run order:
1. Run the install cell below
2. Mount Google Drive and set the config cell
3. Run the import and helper cells
4. Run the final transcription cell


In [ ]:
# Install Colab dependencies
%pip install -q faster-whisper openai python-dotenv

# Install ffmpeg for preprocessing in the Colab runtime.
!apt-get -qq update
!apt-get -qq install -y ffmpeg


## Drive Setup

Clone or upload this repository to Google Drive first.

Default project root:
- `/content/drive/MyDrive/stt-whisper`


In [ ]:
from pathlib import Path

from google.colab import drive

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/stt-whisper')

drive.mount('/content/drive')
assert DRIVE_PROJECT_ROOT.exists(), f'Google Drive project root not found: {DRIVE_PROJECT_ROOT}'
print(f'DRIVE_PROJECT_ROOT = {DRIVE_PROJECT_ROOT}')


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = DRIVE_PROJECT_ROOT

TRANSCRIBE_PRESET = 'high_quality'  # 'high_quality', 'noise_robust', 'low_hallucination'
OUTPUT_DIR = None  # e.g. 'data/outputs'
WHISPER_DEVICE = None  # None, 'auto', 'cuda', 'cpu'
WHISPER_LANGUAGE = None  # e.g. 'ko'
WHISPER_MODEL_SIZE = None  # e.g. 'large-v3'
WHISPER_MODEL_PATH = None  # e.g. '/content/drive/MyDrive/models/whisper-large-v3-ct2'

SKIP_CORRECTION = True  # Flip to False only after the raw transcript looks good.
CORRECTION_MODEL = 'gpt-5.4-mini'
CHUNK_CHARS = 6000
INSTRUCTIONS_FILE = 'prompts/direct_correction_sermon_cross_texts.md'
CORRECTED_OUTPUT_PATH = None

# Set the OpenAI API key here directly, place OPENAI_API_KEY in PROJECT_ROOT/.env,
# or store it in Colab Secrets as OPENAI_API_KEY.
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / '.env')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '').strip()
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = str(userdata.get('OPENAI_API_KEY') or '').strip()
    except Exception:
        OPENAI_API_KEY = ''

if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
else:
    print('OPENAI_API_KEY not found after checking os.environ, PROJECT_ROOT/.env, and Colab Secrets.')

assert PROJECT_ROOT.exists(), f'PROJECT_ROOT not found: {PROJECT_ROOT}'
assert (PROJECT_ROOT / 'src').exists(), f'src directory not found under: {PROJECT_ROOT}'
assert CHUNK_CHARS > 0, 'CHUNK_CHARS must be a positive integer.'
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'TRANSCRIBE_PRESET = {TRANSCRIBE_PRESET}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')
print(f'WHISPER_DEVICE = {WHISPER_DEVICE}')
print(f'WHISPER_LANGUAGE = {WHISPER_LANGUAGE}')
print(f'WHISPER_MODEL_SIZE = {WHISPER_MODEL_SIZE}')
print(f'WHISPER_MODEL_PATH = {WHISPER_MODEL_PATH}')


In [ ]:
import importlib
import sys

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import transcribe_to_txt

transcribe_to_txt = importlib.reload(transcribe_to_txt)
AVAILABLE_PRESETS = tuple(sorted(transcribe_to_txt.TRANSCRIBE_PRESETS))
assert TRANSCRIBE_PRESET in AVAILABLE_PRESETS, f'Unknown preset: {TRANSCRIBE_PRESET}. Available presets: {AVAILABLE_PRESETS}'

run_transcribe_to_txt = transcribe_to_txt.run_transcribe_to_txt
print(f'Available presets: {AVAILABLE_PRESETS}')
print(f'Selected preset config: {transcribe_to_txt.TRANSCRIBE_PRESETS[TRANSCRIBE_PRESET]}')


## Input Selection

Edit the next code cell to choose exactly which `.mp3` files to process.
Use `AVAILABLE_AUDIO_PATHS` as a reference list, then set `INPUT_AUDIO_PATHS` to the files you want.


In [ ]:
AVAILABLE_AUDIO_PATHS = [str(p) for p in sorted((PROJECT_ROOT / 'data').glob('*.mp3'))]
print(f'AVAILABLE_AUDIO_PATHS = {AVAILABLE_AUDIO_PATHS}')

# Edit this list to process only the files you want.
INPUT_AUDIO_PATHS = [
    str(PROJECT_ROOT / 'data' / '법화경제11강.mp3'),
    str(PROJECT_ROOT / 'data' / '법화경제12강.mp3'),
]

assert INPUT_AUDIO_PATHS, 'No input files selected. Set INPUT_AUDIO_PATHS manually.'
print(f'INPUT_AUDIO_PATHS = {INPUT_AUDIO_PATHS}')


In [ ]:
if CORRECTED_OUTPUT_PATH and len(INPUT_AUDIO_PATHS) != 1:
    raise ValueError('CORRECTED_OUTPUT_PATH can only be set when processing exactly one input file.')

results = []
for input_audio_path in INPUT_AUDIO_PATHS:
    print(f'Running transcription for: {input_audio_path}')
    result = run_transcribe_to_txt(
        input_audio_path=input_audio_path,
        preset_name=TRANSCRIBE_PRESET,
        skip_correction=SKIP_CORRECTION,
        correction_model=CORRECTION_MODEL,
        chunk_chars=CHUNK_CHARS,
        instructions_file=INSTRUCTIONS_FILE,
        corrected_output_path=CORRECTED_OUTPUT_PATH,
        output_dir=OUTPUT_DIR,
        device=WHISPER_DEVICE,
        language=WHISPER_LANGUAGE,
        model_size=WHISPER_MODEL_SIZE,
        model_path=WHISPER_MODEL_PATH,
    )
    results.append(result)
results
